Este cuaderno, como su nombre indica, es para realizar el entrenamiento y evaluación de modelos de vídeo.

Los modelos que vamos a trabajar en este cuaderno son los siguientes:
- *MCG-NJU/videomae-base* --> inspirado en cómo aprender los modelos de lenguaje (escondiendo palabras para que el modelo las adivine). Este modelo, toma un video, oculta el 90% de los píxeles (en forma de cubosde espacio-tiempo) y se fuerza a sí mismo a reconstruir lo que falta viendo solo el 10% restante.
- *facebook/timesformer-base-finetuned-k400* --> este modelo soluciona un problema crítico en este tipo de tareas: aplicar la atención (heredado de los transformers) sin tener que hacerlo a cada píxel (lo cual consume mucha memoria). Este modelo primero mira la relación espacial (píxeles dentro de un mismo frame) y luego la relación temporal (el mismo píxel a lo largo de los diferentes frames).
- *google/vivit-b-16x2-kinetics400* --> es la evolución directa de un ViT clásico, los cuales los hemos trabajado en el apartado anterior. Este modelo divide el vídeo en "Tubelets" o tubos 3D. Extrae un bloque de píxeles que atraviesa varios frames de golpe desde la primera capa.  

In [73]:
!pip install -q decord transformers evaluate

In [74]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
import decord
from tqdm import tqdm
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import evaluate
from transformers import (
    AutoImageProcessor,
    AutoModelForVideoClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    AutoProcessor
)
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

In [75]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Entrenamiento Train/Valid

## 1.1. Selección del modelo y parámetros

In [76]:
# Descomentar el modelo que se quiera entrenar:
#MODELO_ELEGIDO = "videomae"
MODELO_ELEGIDO = "timesformer"
#MODELO_ELEGIDO = "vivit"

In [77]:
# Diccionario de Checkpoints (Hugging Face)
MODEL_ZOO = {
    "videomae": "MCG-NJU/videomae-base",
    "timesformer": "facebook/timesformer-base-finetuned-k400",
    "vivit": "google/vivit-b-16x2-kinetics400"
}

MODEL_CHECKPOINT = MODEL_ZOO[MODELO_ELEGIDO]

In [78]:
# Los modelos de vídeo suelen requerir un tamaño de clip fijo (ej. 16 o 8 frames)
# Para este experimento, extraemos 16 frames por vídeo para videome y timesformers o 8 frames para el vivit

if MODELO_ELEGIDO == "vivit":
  NUM_FRAMES_CLIP = 8
else:
  NUM_FRAMES_CLIP = 16

print(NUM_FRAMES_CLIP)

BATCH_SIZE = 4       # Este valor hay que mantenerlo bajo para evitar Out Of Memory en CUDA

16


In [79]:
# RUTAS
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
#OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned"
OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned_experimentacion"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [80]:
decord.bridge.set_bridge('torch') # Silencia logs y une Decord con PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1.2. Carga y Partición de Datos

In [81]:
print(f"Lanzando experimento temporal con: {MODELO_ELEGIDO.upper()}")
df_train_master = pd.read_csv(CSV_TRAIN_MASTER)

# Limpieza rápida
if "Unnamed: 0" in df_train_master.columns:
    df_train_master.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in df_train_master.columns:
    df_train_master.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
df_train_master["label"] = df_train_master["label"].astype(int)

# 90/10 Split
train_df, val_df = train_test_split(
    df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42
)

Lanzando experimento temporal con: TIMESFORMER


In [82]:
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        val_str = str(val)
        if not val_str.endswith(".mp4"): val_str += ".mp4"
        if "videos/" in val_str:
            ruta_completa = os.path.join(base_path, val_str)
        else:
            ruta_completa = os.path.join(base_path, "videos", val_str)
        rutas.append(ruta_completa)
    df['ruta_absoluta'] = rutas
    return df

train_df = fix_video_paths(train_df.copy(), RUTA_BASE_VIDEOS)
val_df = fix_video_paths(val_df.copy(), RUTA_BASE_VIDEOS)

## 1.3. Extracción de Frames y PyTorch dataset

In [83]:
print(f"Cargando procesador para {MODEL_CHECKPOINT}...")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(MODEL_CHECKPOINT)
else:
  processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando procesador para facebook/timesformer-base-finetuned-k400...


In [84]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [85]:
class VideoClassificationDataset(TorchDataset):
    def __init__(self, df, processor, clip_len=16):
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(self.clip_len, len(vr))
            frames = vr.get_batch(frame_indices).numpy()

            # Pasamos los frames de golpe
            if MODELO_ELEGIDO == "vivit":
                inputs = self.processor([list(frames)], return_tensors="pt")
            else:
                inputs = self.processor(list(frames), return_tensors="pt")

            pixel_values = inputs["pixel_values"][0]

            # Corrección exclusiva para ViViT
            if MODELO_ELEGIDO == "vivit":
              if pixel_values.shape[0] == 3:
                pixel_values = pixel_values.permute(1, 0, 2, 3)

        except Exception as e:
            # Salvavidas Seguro: Creamos un array numpy con ruido gris (128)
            # Esto evita los ceros puros que causan NaNs en las capas de normalización.

            if MODELO_ELEGIDO == "vivit":
                frames_falsos = np.ones((self.clip_len, 224, 224, 3), dtype=np.uint8) * 128
                inputs = self.processor([list(frames_falsos)], return_tensors="pt")
                pixel_values = inputs["pixel_values"][0]
            else:
                pixel_values = torch.zeros((self.clip_len, 3, 224, 224))

            etiqueta = 0 # Asumimos clase 0 para no sesgar

        return {"pixel_values": pixel_values, "label": etiqueta}

train_dataset = VideoClassificationDataset(train_df, processor, clip_len=NUM_FRAMES_CLIP)
valid_dataset = VideoClassificationDataset(val_df, processor, clip_len=NUM_FRAMES_CLIP)

In [86]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

## 1.4. Modelo y Métricas

In [87]:
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

In [88]:
model = AutoModelForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    #num_frames = NUM_FRAMES_CLIP, # Descomentar en casode entrenar modelo vitit
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

[transformers] TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [89]:
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

In [90]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

## 1.5. Entrenamiento

In [91]:
# Hiperparámetros por defecto
learning_rate = 5e-5
num_train_epochs = 5
weight_decay = 0.01
warmup_ratio = 0.0
gradient_acumulation_steps = 2
lr_scheduler_type = 'linear'

# Slow cooker temporal
"""learning_rate=3e-5              # LR más bajito (bajamos de 5e-5 a 3e-5)
weight_decay=0.1                # Regularización altísima (subimos de 0.01 a 0.1)
num_train_epochs=8              # Le damos más tiempo para compensar la lentitud
warmup_ratio=0.1                # Calentamiento suave (10% del inicio) para no chocar
gradient_acumulation_steps = 2
lr_scheduler_type = 'linear'"""

# Gradientes Estables (Batch Size 16)
"""learning_rate = 5e-5
weight_decay = 0.01
num_train_epochs = 7
warmup_ratio = 0
gradient_acumulation_steps = 4
lr_scheduler_type = 'linear'"""

# LR Scheduler Agresivo (cosine)
"""learning_rate = 5e-5
weight_decay = 0.05             # Subimos un poco el weight_decay
num_train_epochs = 6
warmup_ratio = 0.1              # Fundamental para el coseno
gradient_acumulation_steps = 2
lr_scheduler_type = 'cosine'"""

tamaño_lote_efectivo = BATCH_SIZE * gradient_acumulation_steps
pasos_por_epoca = len(train_dataset) // tamaño_lote_efectivo
total_train_steps = pasos_por_epoca * num_train_epochs

warmup_steps_calculados = int(total_train_steps * warmup_ratio)

print(f"🔧 Lote efectivo: {tamaño_lote_efectivo}")
print(f"🔧 Pasos totales estimados: {total_train_steps}")
print(f"🔧 Pasos de calentamiento (warmup): {warmup_steps_calculados}")

🔧 Lote efectivo: 8
🔧 Pasos totales estimados: 1125
🔧 Pasos de calentamiento (warmup): 0


In [92]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # CRUCIAL
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=gradient_acumulation_steps,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    warmup_steps=warmup_steps_calculados,
    #warmup_ratio=warmup_ratio, Deprecado
    lr_scheduler_type=lr_scheduler_type
)

In [93]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [94]:
print(f"\n🚀 Lanzando el entrenamiento temporal ({MODELO_ELEGIDO})...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))
print("✅ ¡Entrenamiento completado!")


🚀 Lanzando el entrenamiento temporal (timesformer)...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.385666,0.699765,0.485112,0.537313
2,0.951034,0.747282,0.552786,0.557214
3,0.363125,1.016961,0.561918,0.562189
4,0.061610,1.685633,0.569743,0.572139
5,0.001095,1.668033,0.600794,0.601990


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Guardando el modelo definitivo...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Resultados contra fichero de test estático

## 2.1. Configuración y rutas


In [95]:
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
#CSV_SALIDA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned/predicciones_{MODELO_ELEGIDO}_test.csv"
CSV_SALIDA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/{MODELO_ELEGIDO.upper()}_FineTuned_experimentacion/predicciones_{MODELO_ELEGIDO}_test.csv"

## 2.2. Carga de los datos (test) y del modelo

In [96]:
print("Cargando el dataset estático de test...")
test_df = pd.read_csv(CSV_TEST)

# Limpieza de columnas igual que en Train
if "Unnamed: 0" in test_df.columns:
    test_df.drop(columns=["Unnamed: 0"], inplace=True)
if "label_task_3_1_merged" in test_df.columns:
    test_df.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
test_df["label"] = test_df["label"].astype(int)

Cargando el dataset estático de test...


In [97]:
# Arreglar rutas absolutas (usamos la misma lógica que tenías)
def fix_video_paths(df, base_path):
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'
    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"): val = str(val) + ".mp4"
        rutas.append(os.path.join(base_path, val))
    df['ruta_absoluta'] = rutas
    return df

test_df = fix_video_paths(test_df, RUTA_BASE_VIDEOS)

In [98]:
# Función de muestreo de frames (se usa en el entrenamiento)
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

In [99]:
print(f"Cargando procesador y modelo {MODELO_ELEGIDO.upper()} entrenado...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if MODELO_ELEGIDO == "vivit":
  processor = AutoProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")
else:
  processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR + "/modelo_final")

model = AutoModelForVideoClassification.from_pretrained(OUTPUT_DIR + "/modelo_final").to(device)
model.eval()

print(f"Iniciando inferencia sobre {len(test_df)} vídeos de test...")

Cargando procesador y modelo TIMESFORMER entrenado...


Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

Iniciando inferencia sobre 502 vídeos de test...


# 2.3. Inferencia directa video a video

In [100]:
y_true = []
y_pred = []
resultados_para_csv = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    ruta_video = row['ruta_absoluta']
    true_label = row['label']

    try:
        # Lectura eficiente con Decord
        vr = VideoReader(ruta_video, ctx=cpu(0))
        total_frames = len(vr)
        frame_indices = sample_frame_indices(NUM_FRAMES_CLIP, total_frames)

        # ⚠️ CORRECCIÓN AQUÍ: Usamos .numpy() en lugar de .asnumpy()
        frames = vr.get_batch(frame_indices).numpy()

        # Procesamos los 16 fotogramas de golpe
        if MODELO_ELEGIDO == "vivit":
          inputs = processor([list(frames)], return_tensors="pt")
        else:
          inputs = processor(list(frames), return_tensors="pt")

        pixel_values = inputs["pixel_values"].to(device)

        # Inferencia
        with torch.no_grad():
            outputs = model(pixel_values=pixel_values)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

    except Exception as e:
        print(f"\n⚠️ Error procesando {ruta_video}: {e}")
        # En caso de vídeo corrupto, asumimos incertidumbre (0.5) para no romper el test
        prob_misogino = 0.5

    # 1 si prob > 0.5, sino 0
    prediccion_binaria = 1 if prob_misogino > 0.5 else 0

    y_true.append(true_label)
    y_pred.append(prediccion_binaria)

    # Guardamos para el Ensemble Multimodal
    resultados_para_csv.append({
        "id_EXIST": id_vid,
        "prob_misogino_video": prob_misogino,
        "prediccion_binaria_video": prediccion_binaria,
        "label_real": true_label
    })

  2%|▏         | 8/502 [00:30<58:44,  7.13s/it]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6927331134334373126.mp4: [18:53:48] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [18:53:48] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KEEP_REF) >= 0 (-22 vs. 0) Error while feeding the filter graph


 16%|█▌        | 78/502 [01:56<06:33,  1.08it/s]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6993013215147855109.mp4: [18:55:14] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


 28%|██▊       | 142/502 [03:00<04:05,  1.47it/s]

...

 94%|█████████▍| 471/502 [08:26<00:29,  1.06it/s]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6957860609790676230.mp4: [19:01:45] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


 98%|█████████▊| 494/502 [08:49<00:06,  1.18it/s]


⚠️ Error procesando /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/6985683837988752646.mp4: [19:02:08] /github/workspace/src/video/ffmpeg/threaded_decoder.cc:292: [19:02:08] /github/workspace/src/video/ffmpeg/filter_graph.cc:100: Check failed: av_buffersrc_add_frame_flags(buffersrc_ctx_, frame, AV_BUFFERSRC_FLAG_KEEP_REF) >= 0 (-22 vs. 0) Error while feeding the filter graph


100%|██████████| 502/502 [08:58<00:00,  1.07s/it]


## 2.4. Guardado del CSV de predicción y muestreo de métricas

In [101]:
# Guardamos el CSV
os.makedirs(os.path.dirname(CSV_SALIDA), exist_ok=True)
pd.DataFrame(resultados_para_csv).to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV de {MODELO_ELEGIDO} guardado para el Ensemble en: {CSV_SALIDA}!")


✅ ¡CSV de timesformer guardado para el Ensemble en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/TIMESFORMER_FineTuned_experimentacion/predicciones_timesformer_test.csv!


In [102]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS TEST ESTÁTICO: {MODELO_ELEGIDO} (Análisis Temporal)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS TEST ESTÁTICO: timesformer (Análisis Temporal)
F1-Score (Macro): 0.5596
Accuracy: 0.5598

Matriz de Confusión:
 [[145 116]
 [105 136]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.58      0.56      0.57       261
    Misógino       0.54      0.56      0.55       241

    accuracy                           0.56       502
   macro avg       0.56      0.56      0.56       502
weighted avg       0.56      0.56      0.56       502

